# 26 — Sequential Simulation

Consolidates legacy experiments 89, 90, and 91.

1. **Single Trotter step** — execute one Trotter step of the target Hamiltonian (Exp 89)
2. **Multi-step Trotter** — sweep number of Trotter steps and characterize (Exp 90)
3. **Trotter error analysis** — extract and plot Trotter error vs. step count (Exp 91)

## 1. Shared Session Bootstrap

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from qualang_tools.units import unit

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "qubox").exists():
    REPO_ROOT = Path(r"E:\qubox")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from qubox.compat.notebook import (
    load_stage_checkpoint,
    open_notebook_stage,
    save_stage_checkpoint,
)

REGISTRY_BASE = Path(r"E:\qubox")
SAMPLE_ID = "post_cavity_sample_A"
COOLDOWN_ID = "cd_2025_02_22"
QOP_IP = "10.157.36.68"
CLUSTER_NAME = "Cluster_2"

stage = open_notebook_stage(
    stage_name="26_sequential_simulation",
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    qop_ip=QOP_IP,
    cluster_name=CLUSTER_NAME,
    force_reopen=True,
    close_existing=True,
)
session = stage.session
attr = stage.attr
SESSION_BOOTSTRAP_PATH = stage.bootstrap_path
u = unit(coerce_to_integer=True)

prev_checkpoint = load_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="25_context_aware_sqr_calibration",
)

print(f"Repo root on sys.path: {REPO_ROOT}")
print(f"Shared session bootstrap: {SESSION_BOOTSTRAP_PATH}")
print(f"Stage checkpoint path: {stage.checkpoint_path}")
print(f"QM endpoint: {QOP_IP} ({CLUSTER_NAME})")
if prev_checkpoint is not None:
    print(f"Prior stage 25 status: {prev_checkpoint['status']}")

## 2. Trotter Simulation Defaults

In [ ]:
TROTTER_N_AVG = 10000
TROTTER_STEP_COUNTS = [1, 2, 4, 8, 16, 32]
TROTTER_DT_NS = 100  # Time per Trotter step

print("Trotter simulation settings:")
print(f"  n_avg: {TROTTER_N_AVG}")
print(f"  step counts: {TROTTER_STEP_COUNTS}")
print(f"  dt per step: {TROTTER_DT_NS} ns")

## 3. Single Trotter Step — Exp 89

Execute a single Trotter step and verify via state measurement.

In [ ]:
single_step_result = session.trotter_single_step(
    dt_ns=TROTTER_DT_NS,
    n_avg=TROTTER_N_AVG,
)

print("Single Trotter step complete.")

## 4. Multi-Step Trotter Sweep — Exp 90

Sweep number of Trotter steps and characterize the resulting state.

In [ ]:
multi_step_results = {}

for n_steps in TROTTER_STEP_COUNTS:
    result = session.trotter_multi_step(
        n_steps=n_steps,
        dt_ns=TROTTER_DT_NS,
        n_avg=TROTTER_N_AVG,
    )
    multi_step_results[n_steps] = result
    print(f"  n_steps={n_steps}: done")

print("Multi-step Trotter sweep complete.")

## 5. Trotter Error Analysis — Exp 91

Extract and plot Trotter error vs. step count.

In [ ]:
# Extract error metric from each multi-step result
n_steps_arr = np.array(TROTTER_STEP_COUNTS)
errors = np.array([
    multi_step_results[n].get("trotter_error", np.nan)
    for n in TROTTER_STEP_COUNTS
])

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(n_steps_arr, errors, "o-", color="C0")
ax.set_xlabel("Number of Trotter steps")
ax.set_ylabel("Trotter error")
ax.set_title("Trotter Error vs. Step Count")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Trotter error analysis complete.")

## 6. Save Checkpoint

In [ ]:
stage_checkpoint_path = save_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="26_sequential_simulation",
    status="characterized",
    summary="Single and multi-step Trotter simulation with error analysis.",
    consumed_inputs={
        "trotter_n_avg": TROTTER_N_AVG,
        "trotter_step_counts": TROTTER_STEP_COUNTS,
        "trotter_dt_ns": TROTTER_DT_NS,
    },
    persisted_outputs={},
    advisory_outputs={},
    next_stage="27_cluster_state_evolution",
    notes=[
        "All Trotter experiments are characterization-only.",
        "Error analysis depends on accurate SQR gates from notebook 25.",
    ],
    metrics={},
)

print(f"Stage checkpoint saved to: {stage_checkpoint_path}")